# BSF MVA pathway — minimal flux model

A stripped-down, interactive version of the
[BSF isoprenoid → carotenoid flux tool](https://beachbio.io/tools/modelling/bsf-carotenoid-flux/).

This notebook keeps **only the mevalonate (MVA) trunk** — the core carbon route that
builds the C5 isoprenoid units and the first prenyl-diphosphates:

```
acetyl-CoA → HMG-CoA → mevalonate → IPP ⇌ DMAPP → GPP → FPP
             (HMGR)    (MVK·PMK·MVD)  (IDI)      (FPPS) (FPPS)
```

Everything the full tool adds — shared cofactor pools, MVK product inhibition, expression
cassettes, the carotenoid tail, and the Monte-Carlo sensitivity ensemble — is **left out on
purpose**. This is the MVP skeleton: seven pools, six reactions, solved to steady state.

Units are **relative** (not µM) — the point is *relative* structure and where flux backs up,
not calibrated titres. Run each cell top to bottom (`Shift+Enter`), then edit the parameters
in the last cell and re-run to see the pathway respond.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Seven pools in the MVA trunk (state vector order)
SPECIES = ["HMG-CoA", "mevalonate", "IPP", "DMAPP", "GPP", "FPP"]

## The rate laws

Each enzyme is a simple **Michaelis–Menten** step with a generic half-saturation constant `Km`.
`V_*` is the maximum turnover of that step (`kcat · [enzyme]`, in relative units) — this is the
single knob per enzyme you'd turn to represent over- or under-expressing it.

- **thiolase + HMGS** — a constant carbon entry (acetyl-CoA held as an unlimited supply here)
- **HMGR** — the classic rate-limiting step, HMG-CoA → mevalonate
- **lower MVA** (MVK·PMK·MVD lumped) — mevalonate → IPP
- **IDI** — reversible IPP ⇌ DMAPP, controlled by an equilibrium constant `Keq`
- **FPPS** — two condensations: IPP+DMAPP → GPP, then GPP+IPP → FPP

Every pool also has a small first-order **dilution** term (`k_dil`) — cell growth plus the
downstream draws (sterols, dolichol, the carotenoid tail…) that the full tool models explicitly.
Here it is lumped into one turnover constant so every pool settles to a finite steady state. The
dilution flux off **FPP** is the model's output — the trunk's throughput to everything downstream.

In [ ]:
def rates(y, p):
    HMG, MEV, IPP, DMAPP, GPP, FPP = y
    Km = p["Km"]
    mm = lambda s, V, K=Km: V * s / (K + s)          # Michaelis-Menten
    bi = lambda a, b, V: V * (a / (Km + a)) * (b / (Km + b))  # ordered bi-substrate

    r = {}
    r["thiolase"] = p["V_thiolase"]                              # → HMG-CoA (constant entry)
    r["hmgr"]     = mm(HMG, p["V_hmgr"])                         # → mevalonate
    r["mva"]      = mm(MEV, p["V_mva"])                          # → IPP
    r["idi"]      = p["V_idi"] * (IPP - DMAPP / p["Keq"]) / (Km + IPP + DMAPP)  # IPP ⇌ DMAPP (signed)
    r["fpps1"]    = bi(DMAPP, IPP, p["V_fpps"])                  # IPP + DMAPP → GPP
    r["fpps2"]    = bi(GPP,   IPP, p["V_fpps"])                  # GPP + IPP  → FPP
    r["output"]   = p["k_dil"] * FPP                            # FPP → everything downstream
    return r


def dydt(t, y, p):
    HMG, MEV, IPP, DMAPP, GPP, FPP = y
    r = rates(y, p)
    kd = p["k_dil"]                                                  # first-order dilution on every pool
    return [
        r["thiolase"] - r["hmgr"]                       - kd * HMG,   # HMG-CoA
        r["hmgr"]     - r["mva"]                        - kd * MEV,   # mevalonate
        r["mva"]      - r["idi"] - r["fpps1"] - r["fpps2"] - kd * IPP,   # IPP
        r["idi"]      - r["fpps1"]                      - kd * DMAPP, # DMAPP
        r["fpps1"]    - r["fpps2"]                      - kd * GPP,   # GPP
        r["fpps2"]    - r["output"],                                 # FPP (output IS its dilution)
    ]

## Solve to steady state

We integrate the ODEs forward until the pools stop changing (`dC/dt ≈ 0`). The FPP **drain
flux** at steady state is the model's output — the throughput of the whole trunk.

In [ ]:
def steady_state(p, t_end=2000.0):
    sol = solve_ivp(dydt, [0, t_end], [0.5] * len(SPECIES), args=(p,),
                    method="LSODA", rtol=1e-8, atol=1e-10, dense_output=True)
    y_ss = sol.y[:, -1]
    return y_ss, rates(y_ss, p), sol


def report(p):
    y_ss, r, _ = steady_state(p)
    print("Steady-state pools (relative units)")
    for name, val in zip(SPECIES, y_ss):
        print(f"  {name:<12} {val:8.3f}")
    print(f"\nPathway output (FPP flux): {r['output']:.3f}")
    return y_ss, r

## The parameters — edit these and re-run

These mirror the full tool's baseline. The biology to try:

- Drop **`V_hmgr`** and watch HMG-CoA back up and output fall — HMGR is the classic bottleneck.
- Drop **`V_mva`** (the lumped lower-MVA step) — mevalonate piles up instead.
- Switch **`Keq`** between the insect value (`0.5`) and a microbial host (`~3`) — this shifts the
  IPP:DMAPP balance that FPPS feeds on.

In [ ]:
params = dict(
    V_thiolase = 2.5,   # acetyl-CoA → HMG-CoA (constant entry)
    V_hmgr     = 1.4,   # HMG-CoA → mevalonate   ← the classic rate-limiting step
    V_mva      = 1.6,   # mevalonate → IPP (MVK·PMK·MVD lumped)
    V_idi      = 3.0,   # IPP ⇌ DMAPP
    V_fpps     = 2.0,   # the two FPPS condensations
    Keq        = 0.5,   # IDI equilibrium: ~0.5 insect, ~3 microbial host
    Km         = 1.0,   # generic half-saturation
    k_dil      = 0.05,  # first-order dilution on every pool (downstream demand + growth)
)

y_ss, r = report(params)

## See it

A quick look at the steady-state pool sizes and the flux through each step.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(SPECIES, y_ss, color="#2a9d8f")
ax1.set_title("Steady-state pools")
ax1.set_ylabel("relative pool size")
ax1.tick_params(axis="x", rotation=45)

flux_names = ["thiolase", "hmgr", "mva", "idi", "fpps1", "fpps2", "output"]
ax2.bar(flux_names, [r[k] for k in flux_names], color="#e76f51")
ax2.set_title("Flux through each step")
ax2.set_ylabel("relative flux")
ax2.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Bonus: is HMGR really the bottleneck?

Sweep the HMGR turnover and watch how the pathway output responds. A step that's limiting shows
a steep response; one that's in excess flattens out. This is the intuition the full tool's
"leverage" bars formalise as flux-control coefficients.

In [ ]:
scan = np.linspace(0.2, 5.0, 40)
out = []
for v in scan:
    p = dict(params, V_hmgr=v)
    _, r_i, _ = steady_state(p)
    out.append(r_i["output"])

plt.figure(figsize=(7, 4))
plt.plot(scan, out, color="#2a9d8f", lw=2)
plt.axvline(params["V_hmgr"], ls="--", color="#888", label="baseline V_hmgr")
plt.xlabel("HMGR turnover (V_hmgr)")
plt.ylabel("pathway output (FPP flux)")
plt.title("Pathway output vs HMGR expression")
plt.legend()
plt.tight_layout()
plt.show()